# MATH 170 — Exam 1 Review (Code Walkthrough)

This notebook mirrors the style of our earlier labs (Python + **NumPy** + **SymPy**) to help you work through the **Exam 1 Review Worksheet** using code.

**How to use this notebook**
- Read the short prompts.
- When you see **GTW (Go To Work!)**, fill in the missing code or complete the task.
- Run cells top-to-bottom.

Topics covered (Chapter 1): **LD (Location/Distance), EF (Essential Functions), LC (Limits/Continuity)**.


## 0. Setup (run once)

In [ ]:
# Standard imports used throughout the course
import numpy as np
import sympy as sp

# Plotting
import matplotlib.pyplot as plt

# Pretty printing for SymPy
sp.init_printing(use_unicode=True)

# Symbol(s) we'll reuse
x, h = sp.symbols('x h', real=True)


---

# Part I — LD: Location and Distance


## 1. Feature spaces (concept)

A feature vector is an ordered list of numbers like:
$$\vec{x} = (x_1, x_2, \dots, x_n).$$

The space is **$\mathbb{R}^n$**, where $n$ is the number of features.

**GTW (Go To Work!):** For the worksheet features (study hours, social apps, sleep, coffee), confirm the dimension using code (just count them!).


In [ ]:
features = ["study_hours", "social_apps", "sleep_hours", "coffee_intake"]
print("Number of features =", len(features))
print("So each student lives in R^n with n =", len(features))


## 2. Euclidean distance in $\mathbb{R}^3$

Worksheet vectors:
$$\vec{x}=(1,-2,3), \quad \vec{y}=(4,0,-1).$$

Distance formula:
$$d(\vec{x},\vec{y})=\sqrt{\sum_{i=1}^n (x_i-y_i)^2}.$$


In [ ]:
x_vec = np.array([1, -2, 3], dtype=float)
y_vec = np.array([4,  0, -1], dtype=float)

dist = np.linalg.norm(x_vec - y_vec)  # Euclidean norm
dist


## 3. k-Nearest Neighbors in 1D (by hand + code)

Training data (x, label):
- (1, M), (3, M), (6, D), (8, D)

Test point: $x=4$

**GTW:** Use code to compute distances, then predict for $k=1$ and $k=3$ using majority vote.


In [ ]:
# Training data
train_x = np.array([1, 3, 6, 8], dtype=float)
train_label = np.array(["M", "M", "D", "D"])

x_test = 4.0

# Compute distances to the test point
dists = np.abs(train_x - x_test)

# Sort by distance
idx = np.argsort(dists)

print("Distances (sorted):")
for i in idx:
    print(f"x={train_x[i]:.0f}, label={train_label[i]}, distance={dists[i]:.0f}")

def predict_knn(k):
    nearest_labels = train_label[idx[:k]]
    # majority vote (ties break by first in sorted order here)
    vals, counts = np.unique(nearest_labels, return_counts=True)
    return vals[np.argmax(counts)], nearest_labels

for k in [1, 3]:
    pred, neigh = predict_knn(k)
    print(f"\nk={k}: neighbors={list(neigh)} -> prediction={pred}")


## 4. Embeddings distance (6D)

Worksheet gives a 6D embedding for the word **flute**:
$$(0.9,1.0,0.7,0.3,0.8,0.9).$$

**GTW:** Create a hypothetical embedding for **horn** (your choice), then compute the Euclidean distance between them.


In [ ]:
flute = np.array([0.9, 1.0, 0.7, 0.3, 0.8, 0.9], dtype=float)

# GTW: choose your own embedding for "horn"
horn = np.array([0.85, 0.95, 0.65, 0.35, 0.75, 0.80], dtype=float)

d_horn_flute = np.linalg.norm(horn - flute)
d_horn_flute


---

# Part II — EF: Essential Functions


## 5. Polynomial: $f(x)=x^3-4x$

You want:
- domain
- roots
- end behavior (limits as $x\to\pm\infty$)
- what degree implies about end behavior


In [ ]:
f = x**3 - 4*x

# Domain for a polynomial is all real numbers
domain_f = sp.S.Reals
domain_f


In [ ]:
# Roots (solve f(x)=0)
sp.factor(f), sp.solve(sp.Eq(f, 0), x)


In [ ]:
# End behavior via limits
lim_pos_inf = sp.limit(f, x, sp.oo)
lim_neg_inf = sp.limit(f, x, -sp.oo)
lim_pos_inf, lim_neg_inf


Optional visualization (to connect shape ↔ algebra).

In [ ]:
# Quick plot
xs = np.linspace(-4, 4, 400)
f_np = sp.lambdify(x, f, "numpy")

plt.figure()
plt.axhline(0)
plt.axvline(0)
plt.plot(xs, f_np(xs))
plt.title(r"Plot of $f(x)=x^3-4x$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.show()


## 6. Rational function
$$g(x)=\frac{x^2+6x+5}{x^2-7x-60}.$$

You want:
- domain (where denominator ≠ 0)
- roots (where numerator = 0 AND in domain)
- end behavior (limit as $x\to\pm\infty$)
- $\lim_{x\to 12} g(x)$ (watch for asymptote vs hole)


In [ ]:
g = (x**2 + 6*x + 5) / (x**2 - 7*x - 60)

# Factor numerator and denominator
num = sp.factor(x**2 + 6*x + 5)
den = sp.factor(x**2 - 7*x - 60)
num, den


In [ ]:
# Domain exclusions: solve denominator = 0
den_zeros = sp.solve(sp.Eq(den, 0), x)
den_zeros


In [ ]:
# Simplify (this reveals cancellation -> hole)
g_simplified = sp.simplify(g)
g_simplified


In [ ]:
# Roots: solve numerator=0, then remove any that are excluded from domain
num_zeros = sp.solve(sp.Eq(num, 0), x)
roots_valid = [r for r in num_zeros if r not in den_zeros]
num_zeros, roots_valid


In [ ]:
# End behavior: limits at infinity
sp.limit(g, x, sp.oo), sp.limit(g, x, -sp.oo)


In [ ]:
# Limit as x -> 12 (vertical asymptote: check one-sided limits)
lim_left_12  = sp.limit(g, x, 12, dir='-')
lim_right_12 = sp.limit(g, x, 12, dir='+')
lim_left_12, lim_right_12


## 7. Exponential: $f(x)=5^x$

You want a table for $x=-2,-1,0,1,2$, interpret what happens when $x$ increases by 1, and compute $\lim_{x\to-\infty} 5^x$.


In [ ]:
# Table
xs = np.array([-2, -1, 0, 1, 2], dtype=float)
vals = 5**xs
list(zip(xs, vals))


In [ ]:
# SymPy limit
sp.limit(5**x, x, -sp.oo)


## 8. Logarithms

Simplify:
$$\ln(x^3 e^2).$$


In [ ]:
expr = sp.log(x**3 * sp.E**2)
sp.simplify(expr)


## 9. Inverse trig

Evaluate:
$$\sin^{-1}\left(\frac12\right).$$


In [ ]:
sp.asin(sp.Rational(1,2))


---

# Part III — LC: Limits and Continuity


## 10. DSP

Evaluate:
$$\lim_{x\to 2}(9x^3-4x+6).$$

Polynomials are continuous everywhere, so DSP works.


In [ ]:
sp.limit(9*x**3 - 4*x + 6, x, 2)


## 11. FERC (difference quotient)

Evaluate:
$$\lim_{h\to 0}\frac{\frac{1}{x+h}-\frac{1}{x}}{h}.$$

This is the derivative of $1/x$ computed from first principles.


In [ ]:
dq = ((1/(x+h)) - (1/x)) / h
dq_simplified = sp.simplify(dq)
dq_simplified


In [ ]:
sp.limit(dq_simplified, h, 0)


## 12. Radical limit (difference quotient)

Find:
$$\lim_{h\to 0}\frac{f(x+h)-f(x)}{h} \quad \text{where } f(x)=\sqrt{3x-2}.$$


In [ ]:
f_rad = sp.sqrt(3*x - 2)
dq2 = (f_rad.subs(x, x+h) - f_rad) / h
dq2


In [ ]:
# Rationalize + simplify, then take the limit
dq2_simplified = sp.simplify(dq2)
dq2_simplified


In [ ]:
sp.limit(dq2_simplified, h, 0)


## 13. Piecewise limits and continuity

\[
f(x)=
\begin{cases}
x+2 & x<1\\
5 & x=1\\
2x & x>1
\end{cases}
\]

Compute left/right limits at 1, the two-sided limit, and compare with $f(1)$.


In [ ]:
f_piece = sp.Piecewise((x+2, x<1), (5, sp.Eq(x,1)), (2*x, x>1))
f_piece


In [ ]:
L_left  = sp.limit(f_piece, x, 1, dir='-')
L_right = sp.limit(f_piece, x, 1, dir='+')
L_two   = sp.limit(f_piece, x, 1)
f1      = f_piece.subs(x, 1)

L_left, L_right, L_two, f1


## 14. Continuity check for
$$f(x)=\frac{x^2-1}{x-1} \text{ at } x=1.$$


In [ ]:
f_rat = (x**2 - 1)/(x - 1)
sp.simplify(f_rat)


In [ ]:
# Limit as x->1 and function value at 1
sp.limit(f_rat, x, 1), f_rat.subs(x, 1)


## 15. IVT (existence of a root)

Determine whether:
$$f(x)=x^3-x-6$$
has a root between $-5$ and $3$. If so, find an interval of width 1 that contains a root.


In [ ]:
f_ivt = x**3 - x - 6

# Check endpoints
f_m5 = sp.N(f_ivt.subs(x, -5))
f_3  = sp.N(f_ivt.subs(x, 3))
f_m5, f_3


In [ ]:
# Search for a width-1 interval [n, n+1] with a sign change
vals = []
for n in range(-5, 3):
    a, b = n, n+1
    fa = float(f_ivt.subs(x, a))
    fb = float(f_ivt.subs(x, b))
    if fa*fb <= 0:
        print(f"[{a},{b}]: f(a)={fa}, f(b)={fb}")


In [ ]:
# Optional: approximate a root numerically (if not exact)
try:
    root_approx = sp.nsolve(f_ivt, 2)  # initial guess near 2
    root_approx
except Exception as e:
    print("nsolve failed:", e)


## 16. Horizontal asymptote

Evaluate:
$$\lim_{x\to\infty}\frac{12x^3+5x^2}{3x-4x^3}.$$


In [ ]:
expr16 = (12*x**3 + 5*x**2) / (3*x - 4*x**3)
sp.limit(expr16, x, sp.oo)


## 17. Vertical asymptote

For:
$$f(x)=\frac{1}{x-3},$$
compute one-sided limits as $x\to 3^\pm$.


In [ ]:
expr17 = 1/(x-3)
sp.limit(expr17, x, 3, dir='-'), sp.limit(expr17, x, 3, dir='+')


## 18. Hyperbolic tangent limits

\[
\tanh(x) = \frac{e^{x}-e^{-x}}{e^{x}+e^{-x}}.
\]

Compute limits as $x\to\infty$ and $x\to-\infty$.


In [ ]:
tanh_expr = (sp.E**x - sp.E**(-x)) / (sp.E**x + sp.E**(-x))
sp.limit(tanh_expr, x, sp.oo), sp.limit(tanh_expr, x, -sp.oo)


---

# Optional: Visualization of the rational function near special points

This helps connect algebra (holes/asymptotes) to graphs.


In [ ]:
# Example: visualize g(x) near x=12 and x=-5
g_np = sp.lambdify(x, g, "numpy")

xs_plot = np.linspace(-15, 20, 2000)
ys_plot = g_np(xs_plot)

plt.figure()
plt.axhline(0)
plt.axvline(12, linestyle='--')
plt.axvline(-5, linestyle='--')
plt.plot(xs_plot, ys_plot)
plt.ylim(-10, 10)  # clip to see behavior without huge spikes
plt.title(r"$g(x)=\frac{x^2+6x+5}{x^2-7x-60}$ (y clipped)")
plt.xlabel("x")
plt.ylabel("g(x)")
plt.show()
